<a href="https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Ready.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Before testing any signal, I looked at the shape of the key fields I'll rely on: impressions_90d, ctr, avg_position, and word_count. Several are heavy-tailed — a small number of pages carry most of the impressions — which means means and correlations can be pulled around by outliers, so I check medians alongside means.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

key_fields = ["impressions_90d", "ctr", "avg_position", "word_count", "days_since_last_update"]
summary = df[key_fields].describe(percentiles=[.5, .9, .99]).T
summary["mean_vs_median_ratio"] = summary["mean"] / summary["50%"]
print(summary[["mean", "50%", "90%", "99%", "max", "mean_vs_median_ratio"]].round(2))

                           mean      50%       90%       99%       max  \
impressions_90d         5200.37   731.00  12136.40  73505.83  517715.0   
ctr                        0.51     0.07      0.65      8.33     100.0   
avg_position              16.34    10.80     36.80     69.90     245.0   
word_count              3107.76  2877.00   5327.00   7292.00    9546.0   
days_since_last_update    46.10    20.00    104.00    106.00     373.0   

                        mean_vs_median_ratio  
impressions_90d                         7.11  
ctr                                     7.30  
avg_position                            1.51  
word_count                              1.08  
days_since_last_update                  2.30  


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal test 1 — "High search volume drives impressions" (from w01 Discovery A).
Signal test 2 — "CTR falls as position worsens" (from w01 Discovery B).
Signal test 3 — "Longer content protects against decline" (from w01 Discovery C).

Verdicts: CONFIRMED / OPPOSITE / MIXED / FALSE, based on what the data actually shows.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Test 1: search_volume -> impressions_90d
corr1 = df["search_volume"].corr(df["impressions_90d"])
verdict1 = "FALSE" if abs(corr1) < 0.1 else "CONFIRMED"
print(f"Test 1 — search_volume vs impressions_90d: corr={corr1:.3f} -> {verdict1}")

# Test 2: CTR by position tier
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
monotonic = ctr_by_pos.is_monotonic_decreasing
verdict2 = "CONFIRMED" if monotonic else "MIXED"
print(f"\nTest 2 — CTR by position tier:\n{ctr_by_pos.round(4)}\n-> {verdict2}")

# Test 3: word_count vs trend_direction
wc = df.groupby("trend_direction")["word_count"].median()
gap = wc["up"] - wc["down"]
verdict3 = "FALSE" if abs(gap) < 200 else "CONFIRMED"
print(f"\nTest 3 — median word_count by trend:\n{wc.round(0)}\ngap(up-down)={gap:.0f} -> {verdict3}")


Test 1 — search_volume vs impressions_90d: corr=0.001 -> FALSE

Test 2 — CTR by position tier:
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554
Name: ctr, dtype: float64
-> CONFIRMED

Test 3 — median word_count by trend:
trend_direction
down      2909.0
flat      2698.0
new       2239.0
stable    2912.0
up        2848.0
Name: word_count, dtype: float64
gap(up-down)=-62 -> FALSE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's hand-written "stale x visible" rule (from w02) assumes staleness (days_since_last_update >= 180) combined with visibility (impressions_90d >= 500) flags pages worth reviewing. The rule's core assumption is that staleness itself associates with decline. I test that assumption directly: does days_since_last_update actually correlate with the declining label?

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
corr_stale = df["days_since_last_update"].corr(df["is_declining_label"])
print(f"Correlation between days_since_last_update and is_declining_label: {corr_stale:.3f}")

stale_decline_rate = df[df["days_since_last_update"] >= 180]["is_declining_label"].mean()
fresh_decline_rate = df[df["days_since_last_update"] < 180]["is_declining_label"].mean()
print(f"Decline rate, stale pages (>=180 days): {stale_decline_rate:.3f}")
print(f"Decline rate, fresh pages (<180 days): {fresh_decline_rate:.3f}")

verdict = "CONFIRMED" if stale_decline_rate > fresh_decline_rate + 0.05 else "MIXED"
print(f"\nVerdict: {verdict}")

Correlation between days_since_last_update and is_declining_label: 0.081
Decline rate, stale pages (>=180 days): 0.471
Decline rate, fresh pages (<180 days): 0.542

Verdict: MIXED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Content teams should not use search volume alone to prioritize pages — it barely relates to actual impressions. Position remains a strong, reliable lever: CTR drops sharply as pages slip down the results, so protecting top positions protects clicks directly. Staleness is directionally associated with decline but is not, on its own, a strong enough signal to be the sole trigger for a review queue — it works better combined with visibility and trend signals, as the ranked model in later weeks will show. These findings are observed and directional on this dataset, not proof of causation.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Summary of verdicts:")
print(f"- Search volume -> impressions:       {verdict1}")
print(f"- CTR declines with worse position:    {verdict2}")
print(f"- Word count protects against decline: {verdict3}")
print(f"- Staleness associates with decline:   {verdict}")

Summary of verdicts:
- Search volume -> impressions:       FALSE
- CTR declines with worse position:    CONFIRMED
- Word count protects against decline: FALSE
- Staleness associates with decline:   MIXED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.